# 🎨 DDIM Interpolation — Masculine Man → Feminine Woman

Implements interpolation **exactly as in the DDIM paper** (Song et al. 2020):

```
man_img  ──VAE encode──► x0_man  ──DDIM invert──► z_T_man  ─┐
                                                              ├─ slerp(α) ──► z_T_interp ──DDIM denoise──► x0_interp ──VAE decode──► image
woman_img ──VAE encode──► x0_woman ──DDIM invert──► z_T_woman ─┘
```

Interpolation happens in **noise space** (z_T ≈ N(0,I)), not VAE latent space,
which avoids the color-fringing artifacts from direct latent lerp/slerp.

**Outputs saved to `OUTPUT_DIR`** — same format as `generate_targets.ipynb`
so `run_interpolation_dps.py` can consume them directly.

## ⚙️ Configuration — edit here

In [ ]:
OUTPUT_DIR       = "SD_cond_SD_controlnet/output/interpolation_experiment"

N_COPIES         = 50    # identical copies per canonical latent (for MMD target)
N_INTERP         = 100   # interpolation steps (including endpoints)
N_DDIM_STEPS     = 50    # DDIM inversion / denoising steps

CONTROLNET_SCALE = 0.8
MAN_SEED         = 0
WOMAN_SEED       = 1
GLOBAL_SEED      = 42

CONTROLNET_MODEL_ID = "xinsir/controlnet-scribble-sdxl-1.0"
SPRINTER_MODEL_ID   = "stabilityai/sdxl-turbo"
ARCHITECT_MODEL_ID  = "stabilityai/stable-diffusion-xl-base-1.0"

MAN_PROMPT = (
    "a hyperrealistic studio portrait photograph of a very masculine man, "
    "strong jawline, short hair, formal attire, sharp features, "
    "professional photography, 8k"
)
WOMAN_PROMPT = (
    "a hyperrealistic studio portrait photograph of a very feminine woman, "
    "long hair, soft features, elegant attire, beautiful, "
    "professional photography, 8k"
)

print(f"OUTPUT_DIR   : {OUTPUT_DIR}")
print(f"N_INTERP     : {N_INTERP}")
print(f"N_DDIM_STEPS : {N_DDIM_STEPS}")
print(f"MAN_SEED={MAN_SEED}  WOMAN_SEED={WOMAN_SEED}")

## 1 · Colab Setup — Login, Clone Repo & Install Packages

In [ ]:
import os
import sys
from huggingface_hub import login
from google.colab import userdata
import wandb

hf_token = userdata.get('HF')
if hf_token:
    login(token=hf_token)
else:
    login()

if 'google.colab' in str(get_ipython()):
    import getpass

    get_ipython().system('pip install -q diffusers transformers accelerate xformers controlnet_aux peft')
    get_ipython().system('pip install -q scikit-learn matplotlib Pillow tqdm')

    github_token = userdata.get('GITHUB')
    token = github_token if github_token else getpass.getpass("GitHub token: ")

    repo_url  = f"https://{token}@github.com/orineo1/conditional-matching-paper.git"
    repo_name = "conditional-matching-paper"
    branch    = "create-interpolate"

    if not os.path.exists(repo_name):
        get_ipython().system(f'git clone {repo_url}')
    else:
        print(f"\u2705 Repo already cloned — pulling latest...")
        get_ipython().system(f'cd {repo_name} && git pull')

    get_ipython().system(f'cd {repo_name} && git checkout {branch}')

    repo_path = f"/content/{repo_name}"
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)
    print(f"\n\u2705 Repo ready. Branch: {branch}")

repo_path = f"/content/{repo_name}/SD_cond_SD_controlnet"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

wandb_token = userdata.get('WANDB')
if wandb_token:
    wandb.login(key=wandb_token)
else:
    wandb.login()

In [ ]:
import os, sys

MODULE_DIR = "/content/conditional-matching-paper/SD_cond_SD_controlnet"
if not os.path.isfile(os.path.join(MODULE_DIR, "models.py")):
    raise FileNotFoundError(f"models.py not found in {MODULE_DIR}. Run the setup cell first.")
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\u2705 Module dir : {MODULE_DIR}")
print(f"\u2705 Output dir : {os.path.abspath(OUTPUT_DIR)}")

## 2 · Imports

In [ ]:
import copy, json, pickle
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm

from models      import load_models
from clip_utils  import load_clip_model, encode_images_clip
from image_utils import build_base_image, sobel_proxy

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\U0001f5a5\ufe0f  Device: {device}")

torch.manual_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

## 3 · Helper Functions

In [ ]:
# ── Image generation + VAE encode ─────────────────────────────────────────────
def generate_one(pipe, prompt, cond_pil, cn_scale, seed):
    """Generate one image, return (PIL, latent [4,64,64] float32 scaled z*sf)."""
    generator = torch.Generator(device=pipe.device).manual_seed(seed)
    with torch.no_grad():
        result = pipe(
            prompt=[prompt], image=[cond_pil],
            num_inference_steps=2, guidance_scale=0.0,
            controlnet_conditioning_scale=cn_scale,
            output_type="pil", return_dict=True, generator=generator,
        )
        pil = result.images[0]
        # VAE encode in fp32 to avoid overflow
        pipe.vae.to(torch.float32)
        img_t = TF.to_tensor(pil).unsqueeze(0).to(pipe.device).float()
        img_t = (img_t * 2.0) - 1.0
        latent = pipe.vae.encode(img_t).latent_dist.mean
        latent = latent * pipe.vae.config.scaling_factor
    if torch.isnan(latent).any() or torch.isinf(latent).any():
        raise RuntimeError(f"NaN/Inf in VAE latent for seed={seed}.")
    return pil, latent[0].float().cpu()


def decode_latents(latents_batch, vae, batch_size=4):
    """Decode [N,4,64,64] float32 scaled latents → list of PIL images."""
    dev = next(vae.parameters()).device
    vae.to(torch.float32)
    images = []
    with torch.no_grad():
        for i in range(0, len(latents_batch), batch_size):
            batch = latents_batch[i:i + batch_size].to(dev).float()
            out   = vae.decode(batch / vae.config.scaling_factor).sample
            out   = torch.clamp((out + 1.0) / 2.0, 0.0, 1.0)
            for j in range(out.shape[0]):
                images.append(TF.to_pil_image(out[j].cpu()))
    return images


def encode_pil_to_clip(pil_list, clip_model, clip_processor, batch_size=8):
    """Encode list of PIL images → [N, 768] float32 CLIP embeddings."""
    all_embs = []
    clip_model.to(device)
    with torch.no_grad():
        for i in range(0, len(pil_list), batch_size):
            batch   = pil_list[i:i + batch_size]
            tensors = torch.cat(
                [TF.to_tensor(img).unsqueeze(0) for img in batch], dim=0
            ).to(device)
            all_embs.append(encode_images_clip(tensors, clip_model, clip_processor).cpu())
    clip_model.to("cpu")
    return torch.cat(all_embs, dim=0).float()


def show_pil_row(images, titles=None, figsize_per=3):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per * n, figsize_per))
    if n == 1: axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, images)):
        ax.imshow(img); ax.axis("off")
        if titles: ax.set_title(titles[i], fontsize=8)
    plt.tight_layout(); plt.show()


# ── DDIM helpers ──────────────────────────────────────────────────────────────
def slerp(v0, v1, t):
    """Spherical linear interpolation between two tensors (any shape)."""
    v0_f = v0.reshape(-1).float()
    v1_f = v1.reshape(-1).float()
    dot   = torch.dot(v0_f / v0_f.norm(), v1_f / v1_f.norm()).clamp(-1, 1)
    omega = torch.acos(dot)
    if omega.abs() < 1e-6:          # nearly identical vectors → plain lerp
        return ((1 - t) * v0 + t * v1).to(v0.dtype)
    return (
        (torch.sin((1 - t) * omega) / torch.sin(omega)) * v0 +
        (torch.sin(      t * omega) / torch.sin(omega)) * v1
    ).to(v0.dtype)


def ddim_invert(x0_latent, unet, scheduler, n_steps,
                encoder_states, added_cond_kwargs):
    """
    DDIM inversion: x0 latent → z_T noise  (deterministic forward ODE).

    Runs the DDIM ODE *backwards* (t=0 → t=T) using the reverse formula:
        x_{t+1} = sqrt(α_{t+1}) * x0_pred  +  sqrt(1-α_{t+1}) * eps_pred

    x0_latent : [1, 4, 64, 64]  float16, scaled (z * sf)
    Returns   : [1, 4, 64, 64]  float16  ≈ N(0, I)
    """
    inv_sched = copy.deepcopy(scheduler)
    inv_sched.set_timesteps(n_steps)
    # flip: go from low t (clean) → high t (noisy)
    timesteps = inv_sched.timesteps.flip(0)
    alphas    = inv_sched.alphas_cumprod.to(x0_latent.device)

    xt = x0_latent.clone()
    with torch.no_grad():
        for idx, t in enumerate(timesteps):
            # predict noise with CFG (guidance=0 → use unconditional half)
            model_input = torch.cat([xt] * 2)
            noise_pred  = unet(
                model_input, t,
                encoder_hidden_states=encoder_states,
                added_cond_kwargs=added_cond_kwargs,
                return_dict=False,
            )[0]
            eps = noise_pred.chunk(2)[0]   # unconditional

            alpha_t = alphas[t.long()]
            # next timestep (one step further into the noisy direction)
            t_next    = timesteps[min(idx + 1, len(timesteps) - 1)]
            alpha_next = alphas[t_next.long()] if idx + 1 < len(timesteps) \
                         else torch.tensor(0.0, device=xt.device)

            # predicted x0
            x0_pred = (xt - (1 - alpha_t).sqrt() * eps) / alpha_t.sqrt()
            # DDIM inversion step
            xt = alpha_next.sqrt() * x0_pred + (1 - alpha_next).sqrt() * eps

    return xt   # z_T


def ddim_denoise(z_T, unet, scheduler, n_steps,
                 encoder_states, added_cond_kwargs):
    """
    DDIM denoising: z_T noise → x0 latent  (deterministic reverse ODE).

    z_T : [1, 4, 64, 64]  float16
    Returns x0 : [1, 4, 64, 64]  float16, scaled (z * sf)
    """
    den_sched = copy.deepcopy(scheduler)
    den_sched.set_timesteps(n_steps)

    xt = z_T.clone()
    with torch.no_grad():
        for t in den_sched.timesteps:
            model_input = torch.cat([xt] * 2)
            noise_pred  = unet(
                model_input, t,
                encoder_hidden_states=encoder_states,
                added_cond_kwargs=added_cond_kwargs,
                return_dict=False,
            )[0]
            eps = noise_pred.chunk(2)[0]   # unconditional
            xt  = den_sched.step(eps, t, xt, return_dict=False)[0]

    return xt   # x0 latent


def plot_pca_interp(coords, n_steps, save_path, title):
    fig, ax = plt.subplots(figsize=(10, 5))
    sc = ax.scatter(coords[:, 0], coords[:, 1],
                    c=np.arange(n_steps), cmap="RdBu_r",
                    s=60, edgecolors="black", linewidths=0.3, zorder=3)
    plt.colorbar(sc, ax=ax, label="Step  (0=man, N-1=woman)")
    ax.plot(coords[:, 0], coords[:, 1], "k--", alpha=0.25, linewidth=1, zorder=2)
    ax.scatter(*coords[0],  s=250, c="royalblue", marker="*", zorder=5, label="Man (α=0)")
    ax.scatter(*coords[-1], s=250, c="crimson",   marker="*", zorder=5, label="Woman (α=1)")
    for idx in range(0, n_steps, max(1, n_steps // 10)):
        ax.annotate(str(idx), coords[idx],
                    textcoords="offset points", xytext=(4, 4), fontsize=7, alpha=0.8)
    ax.set_title(title); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"  \u2705 Saved \u2192 {save_path}")

print("\u2705 Helpers defined.")

## 4 · Load Models

In [ ]:
print("Loading architect + sprinter + ControlNet...")
architect, sprinter = load_models(
    device,
    controlnet_model_id=CONTROLNET_MODEL_ID,
    sprinter_model_id=SPRINTER_MODEL_ID,
    architect_model_id=ARCHITECT_MODEL_ID,
)
print("Loading CLIP...")
clip_model, clip_processor = load_clip_model(device)
clip_model.to("cpu")
print(f"\u2705 Architect scheduler : {type(architect.scheduler).__name__}")
print("\u2705 Models loaded.")

## 5 · Build Base Scribble Conditioning

In [ ]:
base_image_pil, base_tensor = build_base_image(device)
with torch.no_grad():
    sobel_tensor = sobel_proxy(base_tensor, device)
    cond_pil     = T.ToPILImage()(sobel_tensor.squeeze(0).cpu())
cond_pil.save(os.path.join(OUTPUT_DIR, "base_scribble.png"))
show_pil_row([base_image_pil, cond_pil], titles=["Base shape", "Sobel scribble"])
print("\u2705 base_scribble.png saved.")

## 6 · Generate the Two Canonical Portraits

In [ ]:
print(f"Generating canonical man  (seed={MAN_SEED})...")
man_pil, man_latent = generate_one(
    sprinter, MAN_PROMPT, cond_pil, CONTROLNET_SCALE, seed=MAN_SEED
)
man_pil.save(os.path.join(OUTPUT_DIR, "man_canonical.png"))
print(f"  Latent shape : {man_latent.shape}")
print(f"  Latent norm  : {man_latent.norm():.3f}")
show_pil_row([man_pil], titles=[f"Canonical man (seed={MAN_SEED})"])

In [ ]:
print(f"Generating canonical woman (seed={WOMAN_SEED})...")
woman_pil, woman_latent = generate_one(
    sprinter, WOMAN_PROMPT, cond_pil, CONTROLNET_SCALE, seed=WOMAN_SEED
)
woman_pil.save(os.path.join(OUTPUT_DIR, "woman_canonical.png"))
print(f"  Latent shape : {woman_latent.shape}")
print(f"  Latent norm  : {woman_latent.norm():.3f}")
show_pil_row([woman_pil], titles=[f"Canonical woman (seed={WOMAN_SEED})"])

In [ ]:
man_f   = man_latent.reshape(-1).float()
woman_f = woman_latent.reshape(-1).float()
cos_sim = torch.dot(man_f / man_f.norm(), woman_f / woman_f.norm()).item()
l2_dist = (man_latent - woman_latent).norm().item()
print(f"Man \u2194 Woman  cosine sim (VAE latent) : {cos_sim:.4f}")
print(f"Man \u2194 Woman  L2 distance (VAE latent) : {l2_dist:.3f}")
show_pil_row([man_pil, woman_pil], titles=["Man (\u03b1=0)", "Woman (\u03b1=1)"])

## 7 · Prepare DDIM Prompt Embeddings
We use **unconditional** (empty prompt) embeddings for both inversion and denoising,
matching the DDIM paper's setup.

In [ ]:
height, width = 512, 512

with torch.no_grad():
    (prompt_embeds, neg_embeds,
     pooled_embeds, neg_pooled) = architect.encode_prompt(
        prompt="", negative_prompt="",
        device=device, do_classifier_free_guidance=True,
        num_images_per_prompt=1,
    )

add_time_ids = torch.tensor(
    [[height, width, 0, 0, height, width]],
    dtype=prompt_embeds.dtype, device=device
)
added_cond_kwargs = {
    "text_embeds": torch.cat([neg_pooled,  pooled_embeds], dim=0),
    "time_ids":    add_time_ids.repeat(2, 1),
}
encoder_states = torch.cat([neg_embeds, prompt_embeds], dim=0)

print(f"Prompt embeds     : {prompt_embeds.shape}  dtype={prompt_embeds.dtype}")
print(f"Scheduler         : {type(architect.scheduler).__name__}")
print(f"DDIM steps        : {N_DDIM_STEPS}")
print("\u2705 DDIM embeddings ready.")

## 8 · DDIM Inversion — Encode Both Images to Noise Space
Maps each canonical portrait to its corresponding `z_T ≈ N(0,I)` via the
deterministic DDIM forward ODE.

In [ ]:
# Convert our VAE latents (sprinter-scaled) to architect VAE scaling
# Both VAEs are SDXL so scaling_factor should be identical, but we convert
# explicitly to be safe.
sf_s = sprinter.vae.config.scaling_factor
sf_a = architect.vae.config.scaling_factor
print(f"Sprinter  scaling_factor : {sf_s}")
print(f"Architect scaling_factor : {sf_a}")

man_x0   = (man_latent   / sf_s * sf_a).unsqueeze(0).half().to(device)
woman_x0 = (woman_latent / sf_s * sf_a).unsqueeze(0).half().to(device)

print(f"\nInverting man image   ({N_DDIM_STEPS} DDIM steps)...")
z_T_man = ddim_invert(
    man_x0, architect.unet, architect.scheduler,
    N_DDIM_STEPS, encoder_states, added_cond_kwargs
)
print(f"  z_T_man   norm : {z_T_man.norm():.3f}  (expect ~sqrt(4*64*64) ≈ {(4*64*64)**0.5:.0f} for N(0,1))")

print(f"\nInverting woman image ({N_DDIM_STEPS} DDIM steps)...")
z_T_woman = ddim_invert(
    woman_x0, architect.unet, architect.scheduler,
    N_DDIM_STEPS, encoder_states, added_cond_kwargs
)
print(f"  z_T_woman norm : {z_T_woman.norm():.3f}")

# Verify inversion quality: denoise back and check it looks like original
print("\nVerifying inversion (denoise z_T_man → should recover man_pil)...")
x0_rec_man = ddim_denoise(
    z_T_man, architect.unet, architect.scheduler,
    N_DDIM_STEPS, encoder_states, added_cond_kwargs
)
# decode to PIL
architect.vae.to(torch.float32)
with torch.no_grad():
    rec_pix = architect.vae.decode(
        x0_rec_man.float() / sf_a
    ).sample
    rec_pix = torch.clamp((rec_pix + 1.0) / 2.0, 0.0, 1.0)
    man_rec_pil = TF.to_pil_image(rec_pix[0].cpu())

show_pil_row([man_pil, man_rec_pil],
             titles=["Original man_pil", "Reconstructed via DDIM invert→denoise"])
print("\u2705 DDIM inversion complete.")

## 9 · Slerp in Noise Space + DDIM Denoise Each Step
This is the core DDIM interpolation:
1. `slerp(z_T_man, z_T_woman, α)` → interpolated noise vector
2. DDIM denoise → interpolated `x0` latent
3. Store for downstream DPS use

In [ ]:
print(f"Generating {N_INTERP} interpolated latents via DDIM denoise...")
print(f"(Each step runs {N_DDIM_STEPS} DDIM denoising iterations)")
print()

alphas = torch.linspace(0.0, 1.0, N_INTERP)
interp_latents_list = []

for i, alpha in enumerate(tqdm(alphas, desc="DDIM interp")):
    a = alpha.item()

    # 1. Slerp between the two noise vectors in noise space
    z_T_interp = slerp(z_T_man, z_T_woman, a)   # [1, 4, 64, 64] fp16

    # 2. DDIM denoise → x0 latent (in architect VAE scaling)
    x0_interp = ddim_denoise(
        z_T_interp, architect.unet, architect.scheduler,
        N_DDIM_STEPS, encoder_states, added_cond_kwargs
    )

    # 3. Convert to sprinter VAE scaling convention and store as float32
    lat = (x0_interp.squeeze(0).float() / sf_a * sf_s)
    interp_latents_list.append(lat.cpu())

interp_latents = torch.stack(interp_latents_list, dim=0)  # [N_INTERP, 4, 64, 64]

torch.save(interp_latents, os.path.join(OUTPUT_DIR, "interp_vae_latents.pt"))
print(f"\n\u2705 interp_vae_latents.pt \u2192 {tuple(interp_latents.shape)}")
print(f"  norm range : [{interp_latents.norm(dim=(1,2,3)).min():.2f}, "
      f"{interp_latents.norm(dim=(1,2,3)).max():.2f}]")

## 10 · Duplicate Endpoint Latents N_COPIES Times

In [ ]:
# Endpoint latents from the DDIM interpolation (step 0 = man, step N-1 = woman)
man_latents   = interp_latents[0].unsqueeze(0).expand(N_COPIES, -1, -1, -1).clone()
woman_latents = interp_latents[-1].unsqueeze(0).expand(N_COPIES, -1, -1, -1).clone()

torch.save(man_latents,   os.path.join(OUTPUT_DIR, "man_vae_latents.pt"))
torch.save(woman_latents, os.path.join(OUTPUT_DIR, "woman_vae_latents.pt"))
print(f"man_vae_latents.pt   \u2192 {tuple(man_latents.shape)}")
print(f"woman_vae_latents.pt \u2192 {tuple(woman_latents.shape)}")
print(f"\u2705 Saved.")

## 11 · Decode All Interpolation Frames and Save

In [ ]:
print(f"Decoding {N_INTERP} interpolation latents...")
interp_pil = decode_latents(interp_latents, sprinter.vae, batch_size=4)
print(f"\u2705 Decoded {len(interp_pil)} frames.")

decoded_dir = os.path.join(OUTPUT_DIR, "interp_decoded")
os.makedirs(decoded_dir, exist_ok=True)
for i, img in enumerate(tqdm(interp_pil, desc="Saving frames")):
    alpha_str = f"{alphas[i].item():.3f}".replace(".", "p")
    img.save(os.path.join(decoded_dir, f"interp_{i:03d}_a{alpha_str}.png"))
print(f"\u2705 {len(interp_pil)} frames \u2192 {decoded_dir}")

## 12 · Contact Sheet — 10 Keyframes

In [ ]:
keyframe_idx = np.linspace(0, N_INTERP - 1, 10, dtype=int)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("DDIM Interpolation Contact Sheet: Masculine Man \u2192 Feminine Woman",
             fontsize=13, fontweight="bold")
for j, idx in enumerate(keyframe_idx):
    ax = axes[j // 5][j % 5]
    ax.imshow(interp_pil[idx])
    ax.set_title(f"step {idx}  \u03b1={alphas[idx]:.2f}", fontsize=9)
    ax.axis("off")
plt.tight_layout()

contact_path = os.path.join(OUTPUT_DIR, "interp_contact_sheet.png")
fig.savefig(contact_path, dpi=110, bbox_inches="tight")
plt.show()
print(f"\u2705 Contact sheet \u2192 {contact_path}")

## 13 · PCA Visualizations

In [ ]:
print("Fitting PCA on flattened VAE latents...")
all_flat   = interp_latents.reshape(N_INTERP, -1).numpy()
pca_lat    = PCA(n_components=2)
lat_coords = pca_lat.fit_transform(all_flat)
var_lat    = pca_lat.explained_variance_ratio_.sum()
print(f"  Variance explained: {var_lat:.1%}")

plot_pca_interp(
    lat_coords, N_INTERP,
    save_path=os.path.join(OUTPUT_DIR, "interp_viz_latent_pca.png"),
    title=f"VAE Latent PCA (DDIM interp) \u2014 {N_INTERP} steps  Var: {var_lat:.1%}",
)

In [ ]:
print("Encoding all decoded interp frames through CLIP...")
clip_embs   = encode_pil_to_clip(interp_pil, clip_model, clip_processor)
pca_clip    = PCA(n_components=2)
clip_coords = pca_clip.fit_transform(clip_embs.numpy())
var_clip    = pca_clip.explained_variance_ratio_.sum()
print(f"  Variance explained: {var_clip:.1%}")

plot_pca_interp(
    clip_coords, N_INTERP,
    save_path=os.path.join(OUTPUT_DIR, "interp_viz_clip_pca.png"),
    title=f"CLIP PCA (perceptual, DDIM interp) \u2014 {N_INTERP} steps  Var: {var_clip:.1%}",
)

## 14 · Save PCA Models + Metadata

In [ ]:
with open(os.path.join(OUTPUT_DIR, "pca_latent.pkl"), "wb") as f:
    pickle.dump(pca_lat, f)
with open(os.path.join(OUTPUT_DIR, "pca_clip.pkl"), "wb") as f:
    pickle.dump(pca_clip, f)
print("\u2705 PCA models saved.")

metadata = {
    "method":              "ddim_inversion_slerp",
    "n_ddim_steps":        N_DDIM_STEPS,
    "n_interp":            N_INTERP,
    "n_copies":            N_COPIES,
    "man_seed":            MAN_SEED,
    "woman_seed":          WOMAN_SEED,
    "controlnet_scale":    CONTROLNET_SCALE,
    "man_latent_norm":     man_latent.norm().item(),
    "woman_latent_norm":   woman_latent.norm().item(),
    "z_T_man_norm":        z_T_man.norm().item(),
    "z_T_woman_norm":      z_T_woman.norm().item(),
    "pca_latent_var":      float(var_lat),
    "pca_clip_var":        float(var_clip),
    "man_prompt":          MAN_PROMPT,
    "woman_prompt":        WOMAN_PROMPT,
    "outputs": {
        "man_canonical":      "man_canonical.png",
        "woman_canonical":    "woman_canonical.png",
        "man_vae_latents":    "man_vae_latents.pt",
        "woman_vae_latents":  "woman_vae_latents.pt",
        "interp_vae_latents": "interp_vae_latents.pt",
        "interp_decoded":     "interp_decoded/",
        "contact_sheet":      "interp_contact_sheet.png",
        "latent_pca":         "interp_viz_latent_pca.png",
        "clip_pca":           "interp_viz_clip_pca.png",
    }
}
with open(os.path.join(OUTPUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print("\u2705 metadata.json saved.")

## 15 · Summary

In [ ]:
abs_out = os.path.abspath(OUTPUT_DIR)
print("=" * 65)
print("SUMMARY  (DDIM inversion + slerp interpolation)")
print("=" * 65)
print(f"  Method          : DDIM inversion \u2192 slerp in noise space \u2192 DDIM denoise")
print(f"  DDIM steps      : {N_DDIM_STEPS}")
print(f"  Interp steps    : {N_INTERP}")
print(f"  N copies        : {N_COPIES}")
print(f"  z_T_man  norm   : {z_T_man.norm():.3f}")
print(f"  z_T_woman norm  : {z_T_woman.norm():.3f}")
print(f"  VAE latent PCA  : {var_lat:.1%} variance explained")
print(f"  CLIP PCA        : {var_clip:.1%} variance explained")
print(f"  Output dir      : {abs_out}")
print()
print("  Files written:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        print(f"    {fname:<38}  {os.path.getsize(fpath)/1024:7.1f} KB")
    else:
        print(f"    {fname:<38}  [{len(os.listdir(fpath))} files]")
print("=" * 65)
print("\n\u2705 Done! Pass --targets_dir", OUTPUT_DIR, "to run_interpolation_dps.py")